In [26]:
import torch
import torchaudio
import soundfile as sf
import numpy as np
import librosa
import random
from pydub import AudioSegment

# 1. Load data

In [27]:
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

audio_file = "sample_audio.wav"
audio, sr = librosa.load(audio_file, sr=None)
clip = AudioSegment.from_wav(audio_file)

# 2. Export Function

In [28]:
def export_pydub(audio):
    while True:
        file_name = input("input file name: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("File name must ends with .wav")
    audio.export(file_name, format="wav")
    print(f"Audio saved as: {file_name} \n")

def export_librosa(audio):
    while True:
        file_name = input("input file name: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("File name must ends with .wav")
    sf.write(file_name, np.clip(audio, -1, 1), sr)
    print(f"Audio saved as: {file_name} \n")

# 3. Predict Text wav2vec

In [29]:
def predict_text(audio_file):
    audio_np, _ = librosa.load(audio_file, sr=bundle.sample_rate, mono=True)
    waveform = torch.from_numpy(audio_np).unsqueeze(0)

    # waveform, sr = torchaudio.load(audio_file)

    # if sr != bundle.sample_rate:
    #     waveform = torchaudio.functional.resample(waveform, sr, bundle.sample_rate)

    with torch.inference_mode():
        emission, _ = model(waveform)

    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0

        def forward(self, emission: torch.Tensor)->str:
            indices = torch.argmax(emission, dim=-1)
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            return "".join([self.labels[i] for i in indices])

    decoder = GreedyCTCDecoder(bundle.get_labels())
    transcript = decoder(emission[0])
    transcript = transcript.replace("|", " ")
    return transcript

# 4. Augmentation

In [30]:
# augmentation ada 9
# 1. adjvolume
def adjvolume(clip):
    while True:
        nilai = float(input("Input nilai (1 to 5):"))
        if not (1 <= nilai <= 5):
            print("Must be between 1 to 5")
        else:
            break
    export_pydub(clip + nilai)

# 2. repeat audio
def repeat_audio(clip):
    while True:
        count = int(input("Input nilai (Must be more than 0):"))
        if count <= 0 :
            print("Must be more than 0")
        else:
            break
    export_pydub(clip * count)

# 3. fade in
def fade_in(clip):
    while True:
        nilai = int(input("Input nilai (Must be more than 0):"))
        if nilai <= 0 :
            print("Must be more than 0")
        else:
            break
    export_pydub(clip.fade_in(nilai))

# # kalo random
# def fade_in(clip):
#     nilai = random.randint(1, 5)
#     export_pydub(clip.fade_in(clip * 1000))

# 4. fade out
def fade_out(clip):
    while True:
        nilai = int(input("Input nilai (Must be more than 0):"))
        if nilai <= 0 :
            print("Must be more than 0")
        else:
            break
    export_pydub(clip.fade_out(nilai))

# # kalo random
# def fade_out(clip):
#     nilai = random.randint(1, 5)
#     export_pydub(clip.fade_out(clip * 1000)) # diminta dalam second

# 5. reverse audio
def reverse_audio(clip):
    export_pydub(clip.reverse())

# 6. adjpitch
def adjpitch():
    while True:
        pitch_rate = float(input("Input pitch rate (-5 to 5):"))
        if not (-5 <= pitch_rate <= 5):
            print("Must be between -5 to 5")
        else:
            break
    shifted_audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=pitch_rate)
    export_librosa(shifted_audio)

# 7. ajdspeed
def adjspeed():
    while True:
        pitch_rate = float(input("Input pitch rate (1 to 5):"))
        if not (1 <= pitch_rate <= 5):
            print("Must be between 1 to 5")
        else:
            break
    speed_audio = librosa.effects.time_stretch(audio, rate=pitch_rate)
    export_librosa(speed_audio)

# 8. white noise
def white_noise():
    while True:
        level = float(input("Input level (0.1 to 0.5):"))
        if not (0.1 <= level <= 0.5):
            print("Must be between 0.1 to 0.5")
        else:
            break
    noise = np.random.normal(0, audio.std(), audio.size)
    export_librosa(audio + noise * level)

# 9. random gain
def random_gain():
    nilai = np.random.uniform(0.5, 1.5)
    print(f"Random gain x {nilai:.2f}")
    export_librosa(audio * nilai)

# 5. aug menu

In [31]:
def aug_menu():
    while True:
        print("1.	Pitch shift")
        print("2.	Add white noise")
        print("3.	Repeat clip")
        print("4.	Insert fade in")
        print("5.	Insert fade out")
        print("6.	Back to main menu\n")

        choice = input(">> ")
        if choice == "1":
            adjpitch()
        elif choice == "2":
            white_noise()
        elif choice == "3":
            repeat_audio(clip)
        elif choice == "4":
            fade_in(clip)
        elif choice == "5":
            fade_out(clip)
        elif choice == "6":
            break
        else:
            print("Invalid input")

# 6. Main Menu

In [32]:
def main_menu():
    while True:
        print("1. Audio Augmentation")
        print("2. Predict Text")
        print("3. Exit")

        choice = input(">> ")
        if choice == "1":
            print("\nAudio Augmentation Now")
            aug_menu()
        elif choice == "2":
            transcript = predict_text(audio_file)
            print("Hasil Transcript: ")
            print(transcript)
        elif choice == "3":
            break
        else:
            print("Invalid input")

    print("Main Menu Closed")

main_menu()

1. Audio Augmentation
2. Predict Text
3. Exit

Audio Augmentation Now
1.	Pitch shift
2.	Add white noise
3.	Repeat clip
4.	Insert fade in
5.	Insert fade out
6.	Back to main menu

Audio saved as: pitch5.wav 

1.	Pitch shift
2.	Add white noise
3.	Repeat clip
4.	Insert fade in
5.	Insert fade out
6.	Back to main menu

Invalid input
1.	Pitch shift
2.	Add white noise
3.	Repeat clip
4.	Insert fade in
5.	Insert fade out
6.	Back to main menu

1. Audio Augmentation
2. Predict Text
3. Exit
Main Menu Closed
